In [1]:
from dotenv import load_dotenv
from src.db import init_mongo
import os

# === Initialize MongoDB ===
load_dotenv()
uri = os.getenv("MONGODB_URI")
mongo_client = init_mongo()
db = mongo_client["KB_PROPERTY_LAW"]
documents_col = db["documents"]
sections_col = db["legal_sections"]
process_sections_col = db["processed_legal_sections"]
relations_col = db["relations"]
concepts_col = db["concepts"]
triplets_col = db["triplets"]

You successfully connected to MongoDB!


In [16]:
keep_so_hieu = [
    "31/2024/QH15",
    "101/2024/NĐ-CP",
    "102/2024/NĐ-CP",
    "103/2024/NĐ-CP",
    "226/2025/NĐ-CP",
    "27/2023/QH15",
    "95/2024/NĐ-CP",
    "29/2023/QH15",
    "96/2024/NĐ-CP",
    "91/2015/QH13",
    "52/2014/QH13"
]

def cleanup_database(db):
    # Get collections
    concepts = db['concepts']  # Update collection name if different
    relations = db['relations']  # Update collection name if different
    triplets = db['triplets']  # Update collection name if different

    print("Starting cleanup process...")

    # Step 1: Clean up concepts collection
    print("\n1. Cleaning concepts collection...")
    concept_update_count = 0
    for concept in concepts.find({"documents": {"$exists": True}}):
        original_docs = concept.get('documents', [])
        filtered_docs = [doc for doc in original_docs if doc.get('so_hieu') in keep_so_hieu]

        if len(filtered_docs) != len(original_docs):
            concepts.update_one(
                {"_id": concept["_id"]},
                {"$set": {"documents": filtered_docs}}
            )
            concept_update_count += 1

    print(f"   Processed {concept_update_count} concepts")

    # Step 2: Clean up relations collection
    print("\n2. Cleaning relations collection...")
    relation_update_count = 0
    for relation in relations.find({"documents": {"$exists": True}}):
        original_docs = relation.get('documents', [])
        filtered_docs = [doc for doc in original_docs if doc.get('so_hieu') in keep_so_hieu]

        if len(filtered_docs) != len(original_docs):
            relations.update_one(
                {"_id": relation["_id"]},
                {"$set": {"documents": filtered_docs}}
            )
            relation_update_count += 1

    print(f"   Processed {relation_update_count} relations")

    # Step 3: Clean up triplets collection
    print("\n3. Cleaning triplets collection...")
    triplet_delete_count = 0
    for triplet in triplets.find({"documents": {"$exists": True}}):
        original_docs = triplet.get('documents', [])
        filtered_docs = [doc for doc in original_docs if doc.get('so_hieu') in keep_so_hieu]

        if len(filtered_docs) != len(original_docs):
            if filtered_docs:
                triplets.update_one(
                    {"_id": triplet["_id"]},
                    {"$set": {"documents": filtered_docs}}
                )
            else:
                triplets.delete_one({"_id": triplet["_id"]})
            triplet_delete_count += 1
    print(f"   Processed {triplet_delete_count} triplets")


    # Step 4: Remove empty concepts (no documents or empty documents array)
    print("\n4. Removing empty concepts...")
    empty_concepts = concepts.delete_many({
        "$or": [
            {"documents": {"$exists": False}},
            {"documents": {"$size": 0}}
        ]
    }).deleted_count

    print(f"   Deleted {empty_concepts} empty concepts")

    # Step 5: Remove empty relations (no documents or empty documents array)
    print("\n5. Removing empty relations...")
    empty_relations = relations.delete_many({
        "$or": [
            {"documents": {"$exists": False}},
            {"documents": {"$size": 0}}
        ]
    }).deleted_count

    print(f"   Deleted {empty_relations} empty relations")

    # Step 6: Remove orphaned triplets (referencing deleted concepts or relations)
    print("\n6. Removing orphaned triplets...")
    valid_concept_ids = set(concepts.distinct("_id"))
    valid_relation_ids = set(relations.distinct("_id"))

    orphaned_triplets = triplets.delete_many({
        "$or": [
            {"subject_id": {"$nin": list(valid_concept_ids)}},
            {"object_id": {"$nin": list(valid_concept_ids)}},
            {"relation_id": {"$nin": list(valid_relation_ids)}}
        ]
    }).deleted_count

    print(f"   Deleted {orphaned_triplets} orphaned triplets")

    # Summary
    print("\n" + "="*50)
    print("CLEANUP SUMMARY")
    print("="*50)
    print(f"Concepts updated: {concept_update_count}")
    print(f"Relations updated: {relation_update_count}")
    print(f"Triplets deleted: {triplet_delete_count}")
    print(f"Empty concepts deleted: {empty_concepts}")
    print(f"Empty relations deleted: {empty_relations}")
    print(f"Orphaned triplets deleted: {orphaned_triplets}")
    print("="*50)

    # Final counts
    print("\nFinal collection counts:")
    print(f"Concepts: {concepts.count_documents({})}")
    print(f"Relations: {relations.count_documents({})}")
    print(f"Triplets: {triplets.count_documents({})}")

    print("\nCleanup completed successfully!")

In [17]:
cleanup_database(db)

Starting cleanup process...

1. Cleaning concepts collection...
   Updated 19582 concepts

2. Cleaning relations collection...
   Updated 1762 relations

3. Cleaning triplets collection...
   Processed 31692 triplets

4. Removing empty concepts...
   Deleted 17456 empty concepts

5. Removing empty relations...
   Deleted 1084 empty relations

6. Removing orphaned triplets...
   Deleted 0 orphaned triplets

CLEANUP SUMMARY
Concepts updated: 19582
Relations updated: 1762
Triplets deleted: 31692
Empty concepts deleted: 17456
Empty relations deleted: 1084
Orphaned triplets deleted: 0

Final collection counts:
Concepts: 19225
Relations: 1708
Triplets: 32726

Cleanup completed successfully!
